# Python and CLI for Data Science - Session 15

- *Course*: Python and CLI for Data Science
- *Session*: 15
- *Unit*: Scikit-Learn - Classification

### (Re)sources:
- [Python Data Science Handbook](https://jakevdp.github.io/PythonDataScienceHandbook/index.html) _by Jake VanderPlas (Code released under the MIT License)_


- This section will take a closer took at some classifaction algorithms
    
    *Classification is the task of predicting a discrete class label, which is taken from a finite list of possibilities*

- We will focus on three of the most common classification algorithms
    - Naive Bayes
    - Random Forest
    - Support Vector Machines

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline

## Naive Bayes Classification

- Naive Bayes models are a group of extremely fast and simple classification algorithms that are often suitable for very high-dimensional datasets
- They are useful as a quick-and-dirty baseline for a classification problem

### Mathematical Intution

- Naive Bayes classifiers rely on Bayes's theorem
    - Bayes's theorem describes the relationship of conditional probabilities of statistical quantities

- We're interested in finding the probability of a label $L$ given some observed features $P(L~|~{\rm features})$

- Bayes's theorem tells us how to express this in terms of quantities we can compute more directly:

$$
P(L~|~{\rm features}) = \frac{P({\rm features}~|~L)P(L)}{P({\rm features})}
$$

- $P(L)$ and ${P({\rm features})}$ are given by the data
- We need some model by which we can compute $P({\rm features}~|~L)$ for each label

- Training a model to directly learn $P({\rm features}~|~L)$ is diffcult
- We can make it simpler through the use of some simplifying assumptions about the form of this model


- By making very "naive" assumptions about the model, we can find a rough approximation for each class
- Different types of naive Bayes classifiers rest on different naive assumptions about the data

## Gaussian Naive Bayes

- Perhaps the easiest naive Bayes classifier to understand is Gaussian naive Bayes
- With this classifier, the assumption is that *data from each label is drawn from a simple Gaussian distribution*

In [ ]:
from sklearn.datasets import make_blobs

X, y = make_blobs(100, 2, centers=2, random_state=2, cluster_std=1.5)
plt.scatter(X[:, 0], X[:, 1], c=y, s=50, cmap="RdBu")

- The simplest Gaussian model is to assume that the data is described by a Gaussian distribution with no covariance between dimensions (no interaction between features)
- This model can be fit by computing the mean and standard deviation of the points within each label, which is all we need to define such a distribution

In [ ]:
mus = []
stds = []
for label in range(2):
    mask = y == label
    mus.append(X[mask].mean(0))
    stds.append(X[mask].std(0))
print("Mean values per feature and label:")
print(mus)
print("Standard deviation per feature and label:")
print(stds)

- Let's visualize what our "model" has learned
- The ellipses represent the Gaussian model for each label

In [ ]:
fig, ax = plt.subplots()

ax.scatter(X[:, 0], X[:, 1], c=y, s=50, cmap="RdBu")
xlim = (-8, 8)
ylim = (-15, 5)
xg = np.linspace(xlim[0], xlim[1], 60)
yg = np.linspace(ylim[0], ylim[1], 40)
xx, yy = np.meshgrid(xg, yg)
Xgrid = np.vstack([xx.ravel(), yy.ravel()]).T

for mu, std, color in zip(mus, stds, ["red", "blue"]):
    P = np.exp(-0.5 * (Xgrid - mu) ** 2 / std**2).prod(1)
    ax.contour(xx, yy, P.reshape(xx.shape), levels=[0.01, 0.1, 0.5, 0.9], colors=color, alpha=0.2)

ax.set(xlim=xlim, ylim=ylim)

- We now have a simple recipe to compute the likelihood $P({\rm features}~|~L)$ for any data point
- Thus we can quickly compute the posterior ratio (the probability that a data point belongs to a class)
- This procedure is implemented in Scikit-Learn's `sklearn.naive_bayes.GaussianNB` estimator:

In [ ]:
from sklearn.naive_bayes import GaussianNB

model = GaussianNB()
model.fit(X, y)

- Let's generate some new data, predict the label, and plot this new data to get an idea of where the decision boundary is

In [ ]:
grid = np.meshgrid(np.linspace(*xlim, 50), np.linspace(*ylim, 50))
X_new = np.vstack(list(map(np.ravel, grid))).T
y_new = model.predict(X_new)

plt.scatter(X[:, 0], X[:, 1], c=y, s=50, cmap="RdBu")
lim = plt.axis()
plt.scatter(X_new[:, 0], X_new[:, 1], c=y_new, s=20, cmap="RdBu", alpha=0.25)
plt.axis(lim)

- A nice aspect of this Bayesian formalism is that it naturally allows directly computing a probability for a datapoint
- We can compute using the `predict_proba` method
- These probabilities can be interpreted as a type of model uncertainty

In [ ]:
y_prob = model.predict_proba(X_new)
y_prob[1400:1410].round(2)

## Multinomial Naive Bayes

- Another useful example is multinomial naive Bayes
- It uses the multinomial distribution, which describes the probability of observing counts among a number of categories

- Multinomial naive Bayes is often used in text classification, where the features are related to word counts or frequencies
- We will use the Newsgroups corpus as an example of how to classify documents into categories

In [ ]:
from sklearn.datasets import fetch_20newsgroups

data = fetch_20newsgroups()
data.target_names

- We will select just a few of these categories and download the training and testing sets

In [ ]:
categories = ["talk.religion.misc", "soc.religion.christian", "sci.space", "comp.graphics"]
train = fetch_20newsgroups(subset="train", categories=categories)
test = fetch_20newsgroups(subset="test", categories=categories)
print(train.data[2][48:])

- In order to use this data for machine learning, we need to convert the content of each string into a vector of numbers
- For this we will use the `CountVectorizer`, and create a pipeline that attaches it to a multinomial naive Bayes classifier

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import make_pipeline

model = make_pipeline(CountVectorizer(), MultinomialNB())

- With this pipeline, we can apply the model to the training data and predict labels for the test data

In [ ]:
model.fit(train.data, train.target)
labels = model.predict(test.data)

- Let's take a look at the confusion matrix to evaluate our model

In [ ]:
from sklearn.metrics import confusion_matrix

mat = confusion_matrix(test.target, labels)
sns.heatmap(
    mat.T,
    square=True,
    annot=True,
    fmt="d",
    cbar=False,
    xticklabels=train.target_names,
    yticklabels=train.target_names,
    cmap="Blues",
)
plt.xlabel("true label")
plt.ylabel("predicted label")

- This very simple classifier can successfully separate space discussions from computer discussions
- It gets confused between discussions about religion and discussions about Christianity



- The cool thing here is that we now have the tools to determine the category for *any* string, using the `predict` method of this pipeline

In [ ]:
def predict_category(s, train=train, model=model):
    pred = model.predict([s])
    return train.target_names[pred[0]]


print(predict_category("sending a payload to the ISS"))
print(predict_category("discussing the existence of God"))
print(predict_category("determining the screen resolution"))

## Summary

- Naive Bayes classifiers will generally not perform as well as more complicated models, but have several advantages:

    - They are fast for both training and prediction
    - They provide straightforward probabilistic prediction
    - They are often easily interpretable
    - They have few (if any) tunable parameters


- A naive Bayes classifier is often a good choice as an initial baseline classification
- If it does not perform well, then you can begin exploring more sophisticated models, with some baseline knowledge of how well they should perform

## Exercise

- Try using Gaussian naive Bayes on the news classification dataset and multinomial naive Bayes on our synthetic dataset
- Both models do not work. Why?

In [ ]:
# GaussianNB on news
model = make_pipeline(CountVectorizer(), GaussianNB())
model.fit(train.data, train.target)

In [ ]:
# MultinomialNB on synthetic data
model = MultinomialNB()
model.fit(X, y)

# Decision Trees and Random Forests

- Random forests are an example of an *ensemble* method
- It aggregates the results of a set of simpler estimators
- The sum can be greater than the parts: the accuracy of a majority vote among a number of estimators can end up being better than that of any of the individual estimators doing the voting!

## Motivating Random Forests: Decision Trees

- Random forests are built on decision trees
- Decision trees are an extremely intuitive way to classify or label objects: you simply ask a series of questions designed to zero in on the classification
- For example, a decision tree to classify animals might look something like this

![Decision Tree Example](https://github.com/jakevdp/PythonDataScienceHandbook/blob/master/notebooks/figures/05.08-decision-tree.png?raw=1)

- In a well-constructed tree, each question will cut the number of options by approximately half; which makes the model very efficient


- The trick, of course, comes in deciding which questions to ask at each step
- In machine learning implementations of decision trees, the questions generally take the form of axis-aligned splits in the data
- Each node in the tree splits the data into two groups using a cutoff value within one of the features

In [ ]:
from sklearn.datasets import make_blobs

X, y = make_blobs(n_samples=300, centers=4, random_state=0, cluster_std=1.0)
plt.scatter(X[:, 0], X[:, 1], c=y, s=50, cmap="rainbow")

- A decision tree built on this data will iteratively split the data along an axis according to some quantitative criterion
- At each level, it assigns the label of the new region according to a majority vote of points within it

![Decision Tree Levels](https://github.com/jakevdp/PythonDataScienceHandbook/blob/master/notebooks/figures/05.08-decision-tree-levels.png?raw=1)

- After the first split, every point in the upper branch remains unchanged, so there is no need to further subdivide this branch
- Except for nodes that contain all of one color, at each level *every* region is again split along one of the two features

- This process of fitting a decision tree to our data can be done in Scikit-Learn with the ``DecisionTreeClassifier`` estimator

In [ ]:
from sklearn.tree import DecisionTreeClassifier

tree = DecisionTreeClassifier().fit(X, y)

- Here's a utility function to visualize the classifier

In [ ]:
def visualize_classifier(model, X, y, ax=None, cmap="rainbow"):
    ax = ax or plt.gca()

    # Plot the training points
    ax.scatter(X[:, 0], X[:, 1], c=y, s=30, cmap=cmap, clim=(y.min(), y.max()), zorder=3)
    ax.axis("tight")
    ax.axis("off")
    xlim = ax.get_xlim()
    ylim = ax.get_ylim()

    # fit the estimator
    model.fit(X, y)
    xx, yy = np.meshgrid(np.linspace(*xlim, num=200), np.linspace(*ylim, num=200))
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

    # Create a color plot with the results
    n_classes = len(np.unique(y))
    contours = ax.contourf(xx, yy, Z, alpha=0.3, levels=np.arange(n_classes + 1) - 0.5, cmap=cmap, zorder=1)

    ax.set(xlim=xlim, ylim=ylim)

- Now we can examine what the decision tree classification looks like:

In [ ]:
visualize_classifier(DecisionTreeClassifier(), X, y)

- As the depth increases, we tend to get very strangely shaped classification regions
- It's clear that this is less a result of the true, intrinsic data distribution, and more a result of the particular sampling or noise properties of the data
- That is, this decision tree is clearly overfitting our data

## Ensembles of Estimators: Random Forests

- Multiple overfitting estimators can be combined to reduce the effect of this overfitting
- This ensembling method is called *bagging*
- An ensemble of randomized decision trees is known as a *random forest*


- This type of bagging classification can be done manually using Scikit-Learn's `BaggingClassifier` meta-estimator

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier

tree = DecisionTreeClassifier()
bag = BaggingClassifier(tree, n_estimators=100, max_samples=0.8, random_state=1)

bag.fit(X, y)
visualize_classifier(bag, X, y)

- This example randomizes the data by fitting each estimator with a random subset of 80% of the training points
- (In practice, the randomization happens differently, such that none of the training data is "wasted". Technical details can be found in the [Scikit-Learn documentation](http://scikit-learn.org/stable/modules/ensemble.html#forest))

- An optimized ensemble of randomized decision trees is implemented in the `RandomForestClassifier` estimator, which takes care of all the randomization automatically
- All you need to do is select a number of estimators, and it will fit the ensemble of trees
- The ensemble model is much closer to our intuition about how the parameter space should be split

In [ ]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(n_estimators=100, random_state=0)
visualize_classifier(model, X, y)

## Summary

- Random forests are a powerful method with several advantages:

    - Both training and prediction are very fast, because of the simplicity of the underlying decision trees
    - Both tasks can be straightforwardly parallelized, because the individual trees are entirely independent entities
    - The multiple trees allow for a probabilistic classification: a majority vote among estimators gives an estimate of the probability (accessed in Scikit-Learn with the `predict_proba` method)
    - The nonparametric model is extremely flexible and can thus perform well on tasks that are underfit by other estimators
    
- A primary disadvantage of random forests is that the results are not easily interpretable: that is, if you would like to draw conclusions about the *meaning* of the classification model, random forests may not be the best choice

# Exercise

- Play around with the `n_estimators` and `max_depth` parameters of the `RandomForestClassifier`
- How do the two parameters affect the model?

In [ ]:
# Test different parameterizations here
model = RandomForestClassifier(n_estimators=2, max_depth=2, random_state=0)
visualize_classifier(model, X, y)

# Support Vector Machines

- Support vector machines (SVMs) are a type of discriminative classification algorithm
- Similar to decision trees, it splits the feature space using a (non-linear) plane

## Motivating Support Vector Machines

- SVMs attempt to find a *decision boundary* between two classes, that is *as far away as possible* from any point in either class


- As an example of this, consider the simple case of a classification task in which the two classes of points are well separated
- A linear discriminative classifier would attempt to draw a straight line separating the two sets of data
- However, there may be more than one possible dividing line that can perfectly discriminate between the two classes

In [ ]:
from sklearn.datasets import make_blobs

X, y = make_blobs(n_samples=50, centers=2, random_state=0, cluster_std=0.60)
plt.scatter(X[:, 0], X[:, 1], c=y, s=50, cmap="autumn")

- Let us visualize three *very* different separators, nevertheless, perfectly discriminate between these samples
- Which one should you choose?
- Depending on which you choose, a new data point (e.g., the one marked by the "X" in this plot) will be assigned a different label

In [ ]:
xfit = np.linspace(-1, 3.5)
plt.scatter(X[:, 0], X[:, 1], c=y, s=50, cmap="autumn")
plt.plot([0.6], [2.1], "x", color="black", markeredgewidth=2, markersize=10)

for m, b in [(1, 0.65), (0.5, 1.6), (-0.2, 2.9)]:
    plt.plot(xfit, m * xfit + b, "-k")

plt.xlim(-1, 3.5)

## Support Vector Machines: Maximizing the Margin

- The intuition for SVMs is: we can draw around each line a *margin* of some width, up to the nearest point
- We then select the line with the maximum margin

In [ ]:
xfit = np.linspace(-1, 3.5)
plt.scatter(X[:, 0], X[:, 1], c=y, s=50, cmap="autumn")

for m, b, d in [(1, 0.65, 0.33), (0.5, 1.6, 0.55), (-0.2, 2.9, 0.2)]:
    yfit = m * xfit + b
    plt.plot(xfit, yfit, "-k")
    plt.fill_between(xfit, yfit - d, yfit + d, edgecolor="none", color="lightgray", alpha=0.5)

plt.xlim(-1, 3.5)

### Fitting a Support Vector Machine

- To fit an SVM will use Scikit-Learn's support vector classifier (`SVC`)
- For now, we will use a linear kernel and set the ``C`` parameter to a very large number (we'll discuss the meaning of these in more depth momentarily)

In [ ]:
from sklearn.svm import SVC  # "Support vector classifier"

model = SVC(kernel="linear", C=1e10)
model.fit(X, y)

- We will use a convience function to visualize what is happening

In [ ]:
def plot_svc_decision_function(model, ax=None, plot_support=True):
    """Plot the decision function for a 2D SVC"""
    if ax is None:
        ax = plt.gca()
    xlim = ax.get_xlim()
    ylim = ax.get_ylim()

    # create grid to evaluate model
    x = np.linspace(xlim[0], xlim[1], 30)
    y = np.linspace(ylim[0], ylim[1], 30)
    Y, X = np.meshgrid(y, x)
    xy = np.vstack([X.ravel(), Y.ravel()]).T
    P = model.decision_function(xy).reshape(X.shape)

    # plot decision boundary and margins
    ax.contour(X, Y, P, colors="k", levels=[-1, 0, 1], alpha=0.5, linestyles=["--", "-", "--"])

    # plot support vectors
    if plot_support:
        ax.scatter(
            model.support_vectors_[:, 0],
            model.support_vectors_[:, 1],
            s=300,
            linewidth=1,
            edgecolors="black",
            facecolors="none",
        )
    ax.set_xlim(xlim)
    ax.set_ylim(ylim)

- This is the dividing line that maximizes the margin between the two sets of points
- Notice that a few of the training points just touch the margin: they are circled in the following figure
- These points are the pivotal elements of this fit; they are known as the *support vectors*, and give the algorithm its name

In [ ]:
plt.scatter(X[:, 0], X[:, 1], c=y, s=50, cmap="autumn")
plot_svc_decision_function(model)

- A key to this classifier's success is that for the fit, only the positions of the support vectors matter
- Any points further from the margin that are on the correct side do not modify the fit
- This is exemplified in the next figure

In [ ]:
def plot_svm(N=10, ax=None):
    X, y = make_blobs(n_samples=200, centers=2, random_state=0, cluster_std=0.60)
    X = X[:N]
    y = y[:N]
    model = SVC(kernel="linear", C=1e10)
    model.fit(X, y)

    ax = ax or plt.gca()
    ax.scatter(X[:, 0], X[:, 1], c=y, s=50, cmap="autumn")
    ax.set_xlim(-1, 4)
    ax.set_ylim(-1, 6)
    plot_svc_decision_function(model, ax)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(16, 6))
fig.subplots_adjust(left=0.0625, right=0.95, wspace=0.1)
for axi, N in zip(ax, [60, 120]):
    plot_svm(N, axi)
    axi.set_title("N = {0}".format(N))

- The model does not change when training on 60 or 120 points
- This insensitivity to the exact behavior of distant points is one of the strengths of the SVM model

### Beyond Linear Boundaries: Kernel SVM

- Where SVM can become quite powerful is when it is combined with *kernels*
- Kernels project data into a higher-dimensional space, and thereby allow the model to fit for nonlinear relationships


- Let's look at some data that is not linearly separable

In [ ]:
from sklearn.datasets import make_circles

X, y = make_circles(100, factor=0.1, noise=0.1, random_state=0)

clf = SVC(kernel="linear").fit(X, y)

plt.scatter(X[:, 0], X[:, 1], c=y, s=50, cmap="autumn")
plot_svc_decision_function(clf, plot_support=False)

- But by projecting the data into a higher dimension a linear separator *would* be sufficient
- For example, one simple projection we could use would be to compute a *radial basis function* (RBF) centered on the middle clump

In [ ]:
r = np.exp(-(X**2).sum(1))

- We can visualize this extra data dimension using a three-dimensional plot

In [ ]:
from mpl_toolkits import mplot3d

ax = plt.subplot(projection="3d")
ax.scatter3D(X[:, 0], X[:, 1], r, c=y, s=50, cmap="autumn")
ax.view_init(elev=20, azim=30)
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_zlabel("r")

- With this additional dimension the data becomes trivially linearly separable


- SVM uses a neat little procedure known as the [*kernel trick*](https://en.wikipedia.org/wiki/Kernel_trick)
- A fit on kernel-transformed data can be done implicitly, without ever building the full $N$-dimensional representation of the kernel projection
- This kernel trick is built into the SVM, and is one of the reasons the method is so powerful


- In Scikit-Learn, we can apply kernelized SVM simply by changing our linear kernel to an RBF kernel, using the `kernel` model hyperparameter

In [ ]:
clf = SVC(kernel="rbf", C=1e6)
clf.fit(X, y)

- Using this kernelized support vector machine, we learn a suitable nonlinear decision boundary

In [ ]:
plt.scatter(X[:, 0], X[:, 1], c=y, s=50, cmap="autumn")
plot_svc_decision_function(clf)
plt.scatter(clf.support_vectors_[:, 0], clf.support_vectors_[:, 1], s=300, lw=1, facecolors="none")

### Tuning the SVM: Softening Margins

- Up to now we have considered datasets in which a perfect decision boundary exists
- But what if your data has some amount of overlap?

In [ ]:
X, y = make_blobs(n_samples=100, centers=2, random_state=0, cluster_std=1.2)
plt.scatter(X[:, 0], X[:, 1], c=y, s=50, cmap="autumn")

- SVM has a factor that allows some of the points to creep into the margin if that allows a better fit
- The hardness of the margin is controlled by a tuning parameter `C`
- For a very large `C`, the margin is hard, and points cannot lie in it
- For a smaller `C`, the margin is softer and can grow to encompass some points

In [ ]:
X, y = make_blobs(n_samples=100, centers=2, random_state=0, cluster_std=0.8)

fig, ax = plt.subplots(1, 2, figsize=(16, 6))
fig.subplots_adjust(left=0.0625, right=0.95, wspace=0.1)

for axi, C in zip(ax, [10.0, 0.1]):
    model = SVC(kernel="linear", C=C).fit(X, y)
    axi.scatter(X[:, 0], X[:, 1], c=y, s=50, cmap="autumn")
    plot_svc_decision_function(model, axi)
    axi.scatter(model.support_vectors_[:, 0], model.support_vectors_[:, 1], s=300, lw=1, facecolors="none")
    axi.set_title("C = {0:.1f}".format(C), size=14)

## Summary

- SVMs are a powerful classification method, for a number of reasons:

    - Their dependence on relatively few support vectors means that they are compact and take up very little memory
    - Once the model is trained, the prediction phase is very fast
    - Because they are affected only by points near the margin, they work well with high-dimensional data
    - Their integration with kernel methods makes them very versatile

- However, SVMs have several disadvantages as well

    - The scaling with the number of samples $N$ is $\mathcal{O}[N^2]$ for efficient implementations. For large numbers of training samples, this computational cost can be prohibitive
    - The results are strongly dependent on a suitable choice for the softening parameter `C`. This must be carefully chosen via cross-validation, which can be expensive as datasets grow in size
    - The results do not have a direct probabilistic interpretation. This can be estimated via an internal cross-validation (see the `probability` parameter of `SVC`), but this extra estimation is costly

# Exercise

- Train a classification algorithm on the [Labeled Faces in the Wild](http://vis-www.cs.umass.edu/lfw/) dataset
- It is a collection of 62 by 47 pixel grayscale images famous people (see the plot below)
- The dataset is split into a train and test split and preprocessed using PCA
- An example model using `GaussianNB` is given
- Exchange `GaussianNB` for the `RandomForestClassifier` and `SVC` models on the dataset and try to obtain the highest accuracy on the test set

In [ ]:
from sklearn.datasets import fetch_lfw_people
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

faces = fetch_lfw_people(min_faces_per_person=60)
X_train, X_test, y_train, y_test = train_test_split(faces.data, faces.target, random_state=42)


def plot_faces(y_pred=None):
    fig, ax = plt.subplots(3, 5, figsize=(8, 6))
    if y_pred is not None:
        fig.suptitle(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
    for i, axi in enumerate(ax.flat):
        axi.imshow(X_test[i].reshape(62, 47), cmap="bone")
        axi.set(xticks=[], yticks=[], xlabel=faces.target_names[y_test[i]].split()[-1])
        if y_pred is not None:
            axi.set_ylabel(
                faces.target_names[y_pred[i]].split()[-1],
                color="black" if y_pred[i] == y_test[i] else "red",
            )


plot_faces()

In [ ]:
# Here is an initial first pipeline
# Exchange the GuassianNB model for a more powerful model and 
# try to obtain the highest accuracy on the test test
# Which model with which parameterization works best?
from sklearn.pipeline import make_pipeline
from sklearn.decomposition import PCA

pca = PCA(n_components=150, whiten=True, random_state=42)
gnb = GaussianNB()
model = make_pipeline(pca, gnb)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

plot_faces(y_pred)